# AIDA-X Model Trainer

This notebook trains a neural-network guitar amplifier / pedal model
for the [AIDA-X](https://github.com/AidaDSP/AIDA-X) plugin and the
[aidadsp-lv2](https://github.com/AidaDSP/aidadsp-lv2) suite.

When run in Google Colab it uses the GPU runtime automatically (Runtime →
Change runtime type → T4 / L4 / A100). It also works locally on any machine
that has a recent PyTorch install (CUDA, ROCm or Apple-Silicon MPS).

> **Modernized 2026 edition.** This version is built for the PyTorch / CUDA
> versions currently shipped with Google Colab. It removes the previous
> TensorFlow / Keras export pipeline (the AIDA-X / RTNeural JSON is now
> produced directly from the trained PyTorch model) and drops the forced
> downgrade of PyTorch that broke the notebook every time Colab refreshed
> its base image.

## Workflow
0. **Setup** — install missing deps, clone the trainer, pick a device.
1. **Data** — upload your `input.wav` / `target.wav` pair.
2. **Train** — choose a model size and run the training loop.
3. **Evaluate** — listen to the prediction, look at the waveform overlay.
4. **Export** — download a `.aidax` model file ready to load in the plugin.


## 0. Setup

Run the next three cells once at the start of the session. They will:

1. detect whether you are running in Colab or locally,
2. install only the packages that aren't already present, and
3. clone (or refresh) the trainer source code into the working directory.


In [ ]:
# Detect the runtime and pick a working directory.
import os, sys, subprocess, shutil, importlib

IN_COLAB = 'google.colab' in sys.modules
WORK_DIR = '/content/Automated-GuitarAmpModelling' if IN_COLAB else os.path.abspath('.')
REPO_URL = 'https://github.com/pilali/Automated-GuitarAmpModelling.git'
REPO_BRANCH = 'new'

print('Python   :', sys.version.split()[0])
print('Runtime  :', 'Google Colab' if IN_COLAB else 'local')
print('Work dir :', WORK_DIR)

In [ ]:
# Install only the missing packages. We deliberately do not touch the
# torch / torchaudio / torchvision versions that ship with Colab — the new
# trainer is compatible with PyTorch >= 2.0 across the board.

REQUIRED = {
    'librosa': 'librosa>=0.10',
    'auraloss': 'auraloss>=0.4',
    'scipy': 'scipy>=1.10',
    'plotly': 'plotly>=5.0',
    'tqdm': 'tqdm',
    'tensorboard': 'tensorboard',
}

missing = []
for module_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print('Installing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages already installed.')

In [ ]:
# Clone (or update) the trainer source code.
import subprocess, os

if not os.path.isdir(WORK_DIR):
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, WORK_DIR])
elif not os.path.isdir(os.path.join(WORK_DIR, '.git')):
    print(f'{WORK_DIR} exists but is not a git checkout — using its contents as-is.')
else:
    # Already a git repo: only switch branch when running inside Colab to
    # avoid clobbering a developer's local checkout.
    if IN_COLAB:
        subprocess.check_call(['git', '-C', WORK_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
        subprocess.check_call(['git', '-C', WORK_DIR, 'checkout', REPO_BRANCH])
        subprocess.check_call(['git', '-C', WORK_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH])

os.chdir(WORK_DIR)
for sub in ('Data/train', 'Data/val', 'Data/test', 'Results'):
    os.makedirs(sub, exist_ok=True)
print('Working directory ready at:', os.getcwd())

In [ ]:
# Pick the best available device. The trainer itself does the same
# detection, but having a single `device` variable here makes the
# evaluation cells later in the notebook concise.
import torch

def select_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = select_device()
print('PyTorch         :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device     :', torch.cuda.get_device_name(0))
    print('CUDA version    :', torch.version.cuda)
print('Selected device :', device)

# CUBLAS workspace config makes RNN training deterministic when a seed is set.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:2')

step = 0  # bookkeeping for the asserts at the top of later cells
print('\nSetup OK — proceed to section 1.')

## 1. Data

You need two perfectly synchronised mono WAV files at the same sample rate
(48 kHz is recommended):

* `input.wav` — the dry/DI guitar signal sent into the device you are modelling.
  You can use the reference capture signal provided in `Data/input.wav` in this
  repository.
* `target.wav` — the same signal recorded after going through the amplifier
  or pedal you want to model.

### How to use this section

* **In Colab** — mount your Google Drive and point `DATA_DIR` at the folder
  that contains the two files. Or upload them directly with the second cell.
* **Locally** — set `DATA_DIR` to the absolute path of the folder that holds
  your `input.wav` and `target.wav`.


In [ ]:
# Optional: mount Google Drive (Colab only).
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped:', exc)
else:
    print('Local run — skipping Google Drive mount.')

In [ ]:
# Set this to the directory that contains your input.wav and target.wav.
# Example: '/content/drive/MyDrive/aida-training/MyAmp'
#@markdown * Use the file browser in the left panel to find a folder with your audio, right-click **"Copy Path", paste below**, and run the cell.
#@markdown  * ex. `/content/drive/My Drive/training-data-folder`

DATA_DIR = '' #@param {type: "string"}

# Normalize the target to match the input level (recommended for most rigs).
NORMALIZE = "False" #@param ["False", "True"]

# Automatically detect and correct polarity inversion and small sample-level
# misalignment between input.wav and target.wav. Recommended: leave on. If
# your target is genuinely uncorrelated with the input (very heavy
# saturation), the auto-align step will detect low confidence and leave the
# files untouched.
AUTO_ALIGN = True

# A friendly name for the trained model. Used as a file name prefix.
FILE_NAME = '' #@param {type: "string"}

assert DATA_DIR, 'Please set DATA_DIR to the folder that contains input.wav and target.wav.'
assert os.path.isdir(DATA_DIR), f'DATA_DIR does not exist: {DATA_DIR}'

In [ ]:
# Optional: upload input.wav and/or target.wav directly from your machine
# (Colab only). Skip this cell if your files are already on Drive / disk.
if IN_COLAB and not (os.path.isfile(os.path.join(DATA_DIR, 'input.wav'))
                     and os.path.isfile(os.path.join(DATA_DIR, 'target.wav'))):
    from google.colab import files
    print('Upload input.wav and target.wav:')
    uploaded = files.upload()
    for name, blob in uploaded.items():
        dst = os.path.join(DATA_DIR, name)
        with open(dst, 'wb') as fh:
            fh.write(blob)
        print('Saved', dst)

In [ ]:
# Pre-process the dataset: load, optionally normalize, split into train/val/test.
import librosa, shutil, numpy as np
sys.path.insert(0, WORK_DIR)
from colab_functions import prep_audio, create_csv_nam_v1_1_1, is_ref_input

# If the user is using our reference capture signal, copy it next to the target
# so the input/target alignment can pick the right offsets automatically.
input_path = os.path.join(DATA_DIR, 'input.wav')
target_path = os.path.join(DATA_DIR, 'target.wav')

if not os.path.isfile(input_path):
    ref = os.path.join(WORK_DIR, 'Data', 'input.wav')
    if os.path.isfile(ref):
        shutil.copy(ref, input_path)
        print('Copied reference input.wav into', input_path)
    else:
        raise FileNotFoundError(f'No input.wav in {DATA_DIR} and no reference shipped with the repo.')
assert os.path.isfile(target_path), f'Missing target.wav in {DATA_DIR}'

in_audio, in_sr = librosa.load(input_path, sr=None, mono=True)
tg_audio, tg_sr = librosa.load(target_path, sr=None, mono=True)
assert in_sr == tg_sr, f'Sample rate mismatch: input={in_sr} target={tg_sr}'
if abs(in_audio.size - tg_audio.size) / in_sr > 3.0:
    raise AssertionError(f'Input and target durations differ by more than 3 s ({in_audio.size/in_sr:.2f}s vs {tg_audio.size/tg_sr:.2f}s).')
if in_audio.size != tg_audio.size:
    print(f'Warning: input/target have different lengths ({in_audio.size} vs {tg_audio.size}) — the prep step will trim to the shorter one.')

prep_audio([input_path, target_path], file_name=FILE_NAME, norm=NORMALIZE,
           csv_file=False, auto_align=AUTO_ALIGN)

step = max(step, 1)
print('\nDataset ready in', os.path.join(WORK_DIR, 'Data'))

## 2. Training

Choose a model size and start training. Larger models are more accurate but
use more CPU when running in the plugin in real time.

| Preset            | Architecture       | Approximate CPU on MOD Dwarf |
| ----------------- | ------------------ | ---------------------------- |
| `Lightest`        | LSTM, 8 hidden     | ~25 %                        |
| `Lightest_GRU`    | GRU, 8 hidden      | ~22 %                        |
| `Light`           | LSTM, 12 hidden    | ~30 %                        |
| `Light_GRU`       | GRU, 12 hidden     | ~27 %                        |
| `Light_noFilter`  | LSTM, 12 hidden, no pre-emphasis | ~30 %        |
| `Standard`        | LSTM, 16 hidden    | ~37 %                        |
| `Standard_GRU`    | GRU, 16 hidden     | ~33 %                        |
| `Standard_noFilter` | LSTM, 16 hidden, no pre-emphasis | ~37 %       |
| `Heavy`           | LSTM, 20 hidden    | ~46 %                        |
| `Heavy_GRU`       | GRU, 20 hidden     | ~41 %                        |

Set `EPOCHS` higher (e.g. 600+) for picky amps or high-gain captures.
Early-stopping kicks in after 25 validations without improvement.


In [ ]:
MODEL_TYPE = "Standard" #@param ["Lightest", "Lightest_GRU", "Light", "Light_GRU", "Light_noFilter", "Standard", "Standard_GRU", "Standard_noFilter", "Heavy", "Heavy_GRU"]
SKIP_CONNECTION = True          # almost always wanted
EPOCHS = '' #@param {type: "string"} # 100-2000

CONFIG_MAP = {
    'Lightest':         'LSTM-8',
    'Lightest_GRU':     'GRU-8',
    'Light':            'LSTM-12',
    'Light_GRU':        'GRU-12',
    'Light_noFilter':   'LSTM-12-noFilter',
    'Standard':         'LSTM-16',
    'Standard_GRU':     'GRU-16',
    'Standard_noFilter':'LSTM-16_nofilter',
    'Heavy':            'LSTM-20',
    'Heavy_GRU':        'GRU-20',
}
assert MODEL_TYPE in CONFIG_MAP, f'Unknown MODEL_TYPE {MODEL_TYPE!r}. Pick one of: {list(CONFIG_MAP)}'

CONFIG_FILE = CONFIG_MAP[MODEL_TYPE]
SKIP_CON = 1 if SKIP_CONNECTION else 0
print(f'Will train {MODEL_TYPE} ({CONFIG_FILE}.json) with skip_con={SKIP_CON} for {EPOCHS} epochs.')

In [ ]:
# Run the training script as a subprocess so its stdout streams into the
# notebook cell (you can see the per-epoch ESR live).
assert step >= 1, 'Run section 1 (data preparation) first.'

cmd = [
    sys.executable, 'dist_model_recnet.py',
    '-l', CONFIG_FILE,
    '-fn', FILE_NAME,
    '-sc', str(SKIP_CON),
    '-eps', str(EPOCHS),
    '-dev', 'auto',
]
print('Running:', ' '.join(cmd))
print('-' * 60)

# Stream output line by line.
proc = subprocess.Popen(cmd, cwd=WORK_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training script exited with code {ret}')

model_dir = os.path.join(WORK_DIR, 'Results', f'{FILE_NAME}_{CONFIG_FILE}-{SKIP_CON}')
step = max(step, 2)
print('\nTrained model saved in:', model_dir)

## 3. Evaluation

Look at a slice of the prediction overlaid on the dry / target waveforms,
and listen to all four signals (dry, target, model, error).


In [ ]:
assert step >= 2, 'Run sections 1 and 2 first.'

import numpy as np, IPython, librosa, torch
import plotly.graph_objects as go

from colab_functions import wav2tensor, extract_best_esr_model
from CoreAudioML.networks import load_model
import CoreAudioML.miscfuncs as miscfuncs

model_path, esr = extract_best_esr_model(model_dir)
print(f'Best variant : {os.path.basename(model_path)}  (ESR = {esr:.6f})')

model = load_model(miscfuncs.json_load(model_path)).to(device).eval()

# cuDNN 9.x in current PyTorch refuses LSTM/GRU forward passes longer than
# 65535 samples. Process the audio in chunks under that limit and carry the
# hidden state across chunks so the result is numerically identical to a
# single forward pass.
def predict(model, dry_1d, device, chunk=32768):
    model.reset_hidden()
    out = []
    with torch.no_grad():
        for start in range(0, len(dry_1d), chunk):
            end = min(start + chunk, len(dry_1d))
            x = dry_1d[start:end].to(device).reshape(-1, 1, 1)
            out.append(model(x).cpu().flatten())
            model.detach_hidden()
    model.reset_hidden()
    return torch.cat(out).numpy()

SAMPLERATE = 48000
DURATION = 5.0
samples = int(DURATION * SAMPLERATE)

dry_full = wav2tensor(os.path.join(WORK_DIR, 'Data', 'test', f'{FILE_NAME}-input.wav'))
target_full = wav2tensor(os.path.join(WORK_DIR, 'Data', 'test', f'{FILE_NAME}-target.wav'))
if len(dry_full) <= samples:
    start = 0
    samples = len(dry_full)
else:
    start = np.random.randint(len(dry_full) - samples)
dry = dry_full[start:start + samples]
target = target_full[start:start + samples]
pred = predict(model, dry, device)

viz_samples = min(24000, samples)
times = np.arange(viz_samples) / SAMPLERATE
fig = go.Figure()
fig.add_trace(go.Scatter(x=times, y=dry[:viz_samples].numpy(), name='dry'))
fig.add_trace(go.Scatter(x=times, y=target[:viz_samples].numpy(), name='target'))
fig.add_trace(go.Scatter(x=times, y=pred[:viz_samples], name='prediction'))
fig.update_layout(title='Dry vs Target vs Predicted', xaxis_title='Time (s)', yaxis_title='Amplitude')
fig.show()

print('DRY:')
IPython.display.display(IPython.display.Audio(data=dry.numpy(), rate=SAMPLERATE))
print('TARGET:')
IPython.display.display(IPython.display.Audio(data=target.numpy(), rate=SAMPLERATE))
print('PREDICTION:')
IPython.display.display(IPython.display.Audio(data=pred, rate=SAMPLERATE))
print('DIFFERENCE (target - prediction):')
IPython.display.display(IPython.display.Audio(data=target.numpy() - pred, rate=SAMPLERATE))

step = max(step, 3)

In [ ]:
# Optional: drop in your own dry guitar recordings and listen to the model.
import io

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name, blob in uploaded.items():
        print('###', name)
        dry = wav2tensor(io.BytesIO(blob))
        pred = predict(model, dry, device)
        print('DRY:')
        IPython.display.display(IPython.display.Audio(data=dry.numpy(), rate=SAMPLERATE))
        print('PREDICTION:')
        IPython.display.display(IPython.display.Audio(data=pred, rate=SAMPLERATE))
else:
    print('Set the variable below to a local WAV file path to process it.')
    LOCAL_DRY = ''  # e.g. '/path/to/dry-clip.wav'
    if LOCAL_DRY:
        dry = wav2tensor(LOCAL_DRY)
        pred = predict(model, dry, device)
        IPython.display.display(IPython.display.Audio(data=dry.numpy(), rate=SAMPLERATE))
        IPython.display.display(IPython.display.Audio(data=pred, rate=SAMPLERATE))

## 4. Diagnostic (optional)

Quick sanity-check of the training run and the data:

* training / validation loss curves (log scale),
* final & best test ESR / DC,
* peak and RMS level of the test input and target,
* sample-accurate lag estimate between input and target (cross-correlation).

A target/input lag larger than a handful of samples is **the** most common
reason an LSTM converges to a useless model.


In [ ]:
import json
import numpy as np
from scipy import signal
import plotly.graph_objects as go

stats_path = os.path.join(model_dir, 'training_stats.json')
with open(stats_path) as fp:
    stats = json.load(fp)

train_losses = stats.get('training_losses', [])
val_losses = stats.get('validation_losses', [])
val_x = list(range(2, 2 * len(val_losses) + 1, 2)) if val_losses else []

print(f'Run                : {os.path.basename(model_dir)}')
print(f'Epochs run         : {len(train_losses)}')
print(f'Test ESR (final)   : {stats.get("test_lossESR_final", float("nan")):.6f}')
print(f'Test ESR (best)    : {stats.get("test_lossESR_best",  float("nan")):.6f}')
print(f'Test DC  (final)   : {stats.get("test_lossDC_final",  float("nan")):.6f}')
print(f'Test DC  (best)    : {stats.get("test_lossDC_best",   float("nan")):.6f}')

fig = go.Figure()
if train_losses:
    fig.add_trace(go.Scatter(x=list(range(1, len(train_losses) + 1)), y=train_losses, name='train'))
if val_losses:
    fig.add_trace(go.Scatter(x=val_x, y=val_losses, name='val'))
fig.update_layout(title='Training / validation loss',
                  xaxis_title='Epoch', yaxis_title='Loss (log scale)',
                  yaxis_type='log')
fig.show()

def _db_peak(x):
    return 20 * np.log10(max(float(np.abs(x).max()), 1e-12))

def _db_rms(x):
    return 20 * np.log10(max(float(np.sqrt((x ** 2).mean())), 1e-12))

dry_t = wav2tensor(os.path.join(WORK_DIR, 'Data', 'test', f'{FILE_NAME}-input.wav')).numpy()
tg_t = wav2tensor(os.path.join(WORK_DIR, 'Data', 'test', f'{FILE_NAME}-target.wav')).numpy()
n = min(len(dry_t), len(tg_t))
dry_t, tg_t = dry_t[:n], tg_t[:n]

print()
print(f'Test set: {n / SAMPLERATE:.2f} s of audio')
print(f'  Dry    peak {_db_peak(dry_t):+6.1f} dBFS   rms {_db_rms(dry_t):+6.1f} dBFS')
print(f'  Target peak {_db_peak(tg_t):+6.1f} dBFS   rms {_db_rms(tg_t):+6.1f} dBFS')

# Sample-accurate cross-correlation of the first 5 s (de-meaned to ignore DC).
sec = min(5 * SAMPLERATE, n)
x = dry_t[:sec] - dry_t[:sec].mean()
y = tg_t[:sec] - tg_t[:sec].mean()
xc = signal.correlate(y, x, mode='full', method='fft')
lags = signal.correlation_lags(len(y), len(x), mode='full')
peak_lag = int(lags[np.argmax(np.abs(xc))])
print()
print(f'Estimated target lag vs input on 5 s slice: {peak_lag:+d} samples'
      f' ({peak_lag / SAMPLERATE * 1000:+.2f} ms)')
if abs(peak_lag) > 5:
    print('  WARNING: lag is > 5 samples. This very likely is the reason the model is not learning.')
    print('  Re-align target.wav (sample-accurate) against input.wav before re-training.')
else:
    print('  Alignment looks fine.')

## 5. Export

Produce a `.aidax` JSON file you can load in the
[AIDA-X](https://github.com/AidaDSP/AIDA-X) plugin or in
[aidadsp-lv2](https://github.com/AidaDSP/aidadsp-lv2). The exporter is now
pure-Python — no TensorFlow / Keras needed.


In [ ]:
assert step >= 2, 'Run sections 1 and 2 first.'
from aidax_export import export, _pick_best_model

model_path, esr, input_batch, output_batch = _pick_best_model(model_dir)
model_filename = os.path.basename(model_dir) + '.aidax'
output_path = os.path.join(WORK_DIR if not IN_COLAB else '/content', model_filename)

metadata = {
    'name': FILE_NAME,
    'samplerate': str(SAMPLERATE),
    'source': '',
    'style': '',
    'based': '',
    'author': '',
    'dataset': '',
    'license': '',
    'esr': esr,
}

export(model_path, output_path,
       metadata=metadata,
       input_batch=input_batch,
       output_batch=output_batch)

print('Generated', output_path)

if IN_COLAB:
    from google.colab import files
    files.download(output_path)

step = max(step, 4)